In [2]:
import pandas as pd
import numpy as np

# Load the dataset
# Note: The UCI dataset sometimes uses 'ISO-8859-1' encoding. 
# If you get a UnicodeDecodeError, change it to 'utf-8' or 'cp1252'.
file_path = '../data/raw/online_retail_II.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nData Info:")
df.info()

Dataset Shape: (1067371, 8)

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [3]:
# Check for missing values
print("Missing values per column:\n", df.isnull().sum())

# Check for negative quantities (returns/cancellations)
print("\nNegative quantities:", (df['Quantity'] < 0).sum())

# Check for negative prices
print("Negative prices:", (df['Price'] < 0).sum())

# Check for duplicates
print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
 Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Negative quantities: 22950
Negative prices: 5
Duplicate rows: 34335


In [4]:
# 1. Create a copy to avoid modifying the raw data
df_clean = df.copy()

# 2. Rename columns for easier SQL querying later (no spaces)
df_clean.rename(columns={'Customer ID': 'CustomerID', 'StockCode': 'StockCode'}, inplace=True)

# 3. Drop rows where CustomerID is missing 
# (We cannot calculate user retention without an ID)
df_clean.dropna(subset=['CustomerID'], inplace=True)

# 4. Remove cancelled orders (negative Quantity) and negative prices
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

# 5. Remove duplicates
df_clean.drop_duplicates(inplace=True)

# 6. Fix Data Types (CustomerID is currently a float, make it an integer/string)
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# 7. Convert InvoiceDate to proper datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 8. Create a new column: TotalAmount (Revenue per line item)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['Price']

print("Clean Dataset Shape:", df_clean.shape)
print("\nMissing values after cleaning:\n", df_clean.isnull().sum())

C:\Users\nazmu\AppData\Local\Temp\ipykernel_12848\3707984164.py:5: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.rename(columns={'Customer ID': 'CustomerID', 'StockCode': 'StockCode'}, inplace=True)
C:\Users\nazmu\AppData\Local\Temp\ipykernel_12848\3707984164.py:9: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.dropna(subset=['CustomerID'], inplace=True)
C:\Users\nazmu\AppData\Local\Temp\ipykernel_12848\3707984164.py:15: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace f

Clean Dataset Shape: (779425, 9)

Missing values after cleaning:
 Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
CustomerID     0
Country        0
TotalAmount    0
dtype: int64


In [5]:
# Export to the processed folder
output_path = '../data/processed/clean_retail_data.csv'
df_clean.to_csv(output_path, index=False)
print(f"✅ Clean data saved successfully to {output_path}")

✅ Clean data saved successfully to ../data/processed/clean_retail_data.csv
